# 01 — Demo: RBAC + ABAC + Consumption

The notebook is the *post-talk artifact*. The live demo runs in the
Databricks UI and Power BI; this file mirrors every step in code so the
audience can reproduce it.

**Prereq:** run `00_setup.py` first. You must also have manually created
the workspace groups `admins` and `managers`, and replaced the
placeholder email in `uc_demo.sample.user_region_map`.

## Sections
1. **Grants** — coarse, group-based grants. Boring by design.
2. **ABAC: Tags** — attribute the data
3. **ABAC: Functions** — define how data is transformed
4. **ABAC: Policies** — bind tags + functions + principals
5. **Consumption in Databricks** — same query, different identity, different result
6. **Consumption in Power BI** — connection reference
7. **Cleanup**

In [0]:
CATALOG = "uc_demo"
SCHEMA  = "sample"
spark.sql(f"USE {CATALOG}.{SCHEMA}")

---
## 1. Group-based grants

The "old way." Coarse on/off access at the table level. We grant `SELECT`
broadly so people can *query* the tables — ABAC will then layer fine-grained
restrictions on top.

`account users` is the system group containing every user in the account.
On workspace-only setups it may be called `users` instead — adjust if needed.

In [0]:
%sql
GRANT USE CATALOG ON CATALOG uc_demo            TO `account users`;
GRANT USE SCHEMA  ON SCHEMA  uc_demo.sample     TO `account users`;
GRANT SELECT      ON TABLE   uc_demo.sample.employees TO `account users`;
GRANT SELECT      ON TABLE   uc_demo.sample.customers TO `account users`;
GRANT SELECT      ON TABLE   uc_demo.sample.user_region_map TO `account users`;

In [0]:
%sql
-- The two named groups don't get extra privileges here.
-- Their elevated access comes through ABAC policy *exceptions*.
SHOW GRANTS ON TABLE uc_demo.sample.employees;

### Why this is boring
Every consumer can now `SELECT` everything. To prevent the analyst from
seeing salaries you'd traditionally either:
- Build a view per persona (N views, drift over time), or
- Move data to a separate table (data duplication, lineage break).

ABAC replaces both patterns with one declarative policy.

---
## 2. ABAC — Governed Tags

ABAC policies match against **governed tags** — tags declared at the
metastore level via `CREATE GOVERNED TAG`. Plain `ALTER TABLE … SET TAGS`
writes informational tags that `has_tag_value()` will not see.

Two-step pattern:
1. Declare the governed tag (key, optional allowed values)
2. Apply it to columns/tables with `ALTER … SET TAGS`

Tag scheme:
- `pii` (values: `string`, `numeric`) on `cpr`, `email`, `full_name` → triggers column mask
- `geo_region` (key-only) on `region` columns → triggers row filter

Requires **CREATE** privilege on governed tags. Workspace admins have it by default.

In [0]:
# Step 1: declare the governed tags.
# `CREATE GOVERNED TAG` does not support IF NOT EXISTS, so we wrap in try/except
# to keep this cell idempotent on re-run.

GOVERNED_TAGS = [
    """CREATE GOVERNED TAG pii
         DESCRIPTION 'Marks columns containing personally identifiable information'
         VALUES ('string', 'numeric')""",
    """CREATE GOVERNED TAG geo_region
         DESCRIPTION 'Marks columns used for region-based row filtering'""",
]

for stmt in GOVERNED_TAGS:
    try:
        spark.sql(stmt)
        print(f"[create] {stmt.split()[2]}")
    except Exception as e:
        msg = str(e)
        if "already exists" in msg.lower() or "duplicate" in msg.lower():
            print(f"[skip]   {stmt.split()[2]} (already exists)")
        else:
            raise

In [0]:
%sql
-- Step 2: apply the governed tags to columns
-- (same SET TAGS syntax — but now they're governed because the tag exists at metastore level)
ALTER TABLE uc_demo.sample.employees ALTER COLUMN cpr       SET TAGS ('pii' = 'string');
ALTER TABLE uc_demo.sample.employees ALTER COLUMN email     SET TAGS ('pii' = 'string');
ALTER TABLE uc_demo.sample.employees ALTER COLUMN full_name SET TAGS ('pii' = 'string');

ALTER TABLE uc_demo.sample.customers ALTER COLUMN email     SET TAGS ('pii' = 'string');
ALTER TABLE uc_demo.sample.customers ALTER COLUMN full_name SET TAGS ('pii' = 'string');

ALTER TABLE uc_demo.sample.customers ALTER COLUMN region    SET TAGS ('geo_region');

In [0]:
%sql
-- Verify governed tags exist + are applied
SHOW GOVERNED TAGS;

In [0]:
%sql
-- Verify column-tag bindings
SELECT table_name, column_name, tag_name, tag_value
FROM   system.information_schema.column_tags
WHERE  catalog_name = 'uc_demo'
AND    schema_name  = 'sample'
ORDER  BY table_name, column_name;

---
## 3. ABAC — Functions (UDFs)

Two SQL UDFs encode the actual access logic. They reference
`is_account_group_member()` and `current_user()` to evaluate per-query
who's asking and what they're allowed to see.

### `mask_pii_string`
Returns the original value for `admins`. Otherwise replaces every
alphanumeric character with `X` — preserves shape (e.g. `150789-1234`
becomes `XXXXXX-XXXX`) so the mask is recognisable.

In [0]:
%sql
CREATE OR REPLACE FUNCTION uc_demo.sample.mask_pii_string(val STRING)
RETURNS STRING
RETURN
CASE
    WHEN is_member('admins') THEN val
    WHEN val IS NULL                            THEN NULL
    ELSE regexp_replace(val, '[A-Za-z0-9]', 'X')
END
;

### `region_filter`
Row filter — returns `TRUE` (row visible) when:
- User is in `admins` OR `managers` (no row restriction), OR
- The row's region matches the user's region in `user_region_map`.

In [0]:
%sql
CREATE OR REPLACE FUNCTION uc_demo.sample.region_filter(region STRING)
RETURNS BOOLEAN
RETURN
     is_member('managers')
  OR region IN (
       SELECT m.region
       FROM   uc_demo.sample.user_region_map m
       WHERE  m.user_email = current_user()
     );

---
## 4. ABAC — Policies

Policies bind: **(tag selector) → (UDF) → (principals with optional exceptions)**.

Apply at SCHEMA level so any future table tagged the same way inherits
automatically.

In [0]:
%sql
-- Column mask: any column tagged pii=string is masked for everyone
-- (admins see raw values via logic inside the mask_pii_string UDF)
CREATE OR REPLACE POLICY mask_pii_policy
ON SCHEMA uc_demo.sample
COMMENT 'Mask PII string columns for all but admins'
COLUMN MASK uc_demo.sample.mask_pii_string
TO `account users`
FOR TABLES
MATCH COLUMNS has_tag_value('pii', 'string') AS pii_col
ON COLUMN pii_col;

In [0]:
%sql
-- Row filter: any table with a column tagged geo_region is filtered.
-- admins + managers bypass the filter inside the UDF itself.
CREATE OR REPLACE POLICY filter_region_policy
ON SCHEMA uc_demo.sample
COMMENT 'Filter rows by user region for non-privileged users'
ROW FILTER uc_demo.sample.region_filter
TO `account users`
FOR TABLES
MATCH COLUMNS has_tag('geo_region') AS region_col
USING COLUMNS (region_col);

In [0]:
%sql
-- Confirm both policies registered
SHOW POLICIES ON SCHEMA uc_demo.sample;

---
## 5. Consumption in Databricks

The same SQL produces different results depending on the logged-in user.
On stage, switch browser profiles between identities and re-run.

| Identity | `employees.cpr` | `customers` rows |
| --- | --- | --- |
| `admins` member | `150789-1234` (raw) | All ~1000 rows |
| `managers` member     | `XXXXXX-XXXX`       | All ~1000 rows |
| regular user (in `user_region_map`) | `XXXXXX-XXXX` | Only their region |
| regular user NOT in map | `XXXXXX-XXXX` | 0 rows |

In [0]:
%sql
-- Confirm who is running this query right now
SELECT current_user() AS me, is_account_group_member('admins') AS is_admin,
                           is_account_group_member('managers')     AS is_manager;

In [0]:
%sql
-- PII visibility check (cpr, email)
SELECT emp_id, full_name, email, cpr, salary, country, department
FROM   uc_demo.sample.employees
LIMIT  10;

In [0]:
%sql
-- Row-filter visibility check
SELECT region, COUNT(*) AS row_count
FROM   uc_demo.sample.customers
GROUP  BY region
ORDER  BY region;

---
## 6. Consumption in Power BI

The exact same UC ABAC policies fire when Power BI connects, **provided
PBI authenticates as a real user** (OAuth/SSO). PAT/service-principal
connections give every PBI session one fixed identity.

### Connection (PBI Desktop)
1. **Get Data → Azure → Azure Databricks**
2. **Server hostname:** copy from SQL Warehouse → Connection details
3. **HTTP path:**     copy from same panel
4. **Database:**      `uc_demo`
5. **Auth:** *Microsoft account* (passes user identity to UC) — **not** PAT
6. **Storage mode:** *DirectQuery*. Import bypasses live ABAC enforcement.

---
## 7. Cleanup

Reverses everything created by this notebook. Order matters: drop policies
before functions, untag columns last.

In [0]:
%skip
%sql

-- Drop policies
DROP POLICY mask_pii_policy    ON SCHEMA uc_demo.sample;
DROP POLICY filter_region_policy ON SCHEMA uc_demo.sample;

-- Drop UDFs
DROP FUNCTION IF EXISTS uc_demo.sample.mask_pii_string;
DROP FUNCTION IF EXISTS uc_demo.sample.region_filter;

-- Remove governed tags from columns first
ALTER TABLE uc_demo.sample.employees ALTER COLUMN cpr       UNSET TAGS ('pii');
ALTER TABLE uc_demo.sample.employees ALTER COLUMN email     UNSET TAGS ('pii');
ALTER TABLE uc_demo.sample.employees ALTER COLUMN full_name UNSET TAGS ('pii');
ALTER TABLE uc_demo.sample.customers ALTER COLUMN email     UNSET TAGS ('pii');
ALTER TABLE uc_demo.sample.customers ALTER COLUMN full_name UNSET TAGS ('pii');
ALTER TABLE uc_demo.sample.customers ALTER COLUMN region    UNSET TAGS ('geo_region');

-- Drop the governed tags themselves (metastore-level)
DROP GOVERNED TAG pii;
DROP GOVERNED TAG geo_region;

-- Revoke grants (optional — only run if you want a fully clean slate)
REVOKE SELECT      ON TABLE   uc_demo.sample.employees       FROM `account users`;
REVOKE SELECT      ON TABLE   uc_demo.sample.customers       FROM `account users`;
REVOKE SELECT      ON TABLE   uc_demo.sample.user_region_map FROM `account users`;
REVOKE USE SCHEMA  ON SCHEMA  uc_demo.sample                 FROM `account users`;
REVOKE USE CATALOG ON CATALOG uc_demo                        FROM `account users`;